# 🤟 Wasel v4 Pro — Live Sign Language Translator
### Real-time PSL (Pakistan Sign Language) translation using YOLOv8-Pose + LSTM

**How to use:**
1. Run Cell 1 (Install) → Wait for ✅
2. Run Cell 2 (Setup) → Wait for ✅
3. Run Cell 3 (Launch) → Click the `*.gradio.live` link!

> ⚡ **No Docker. No Cloud Shell. Just run and demo!**

In [ ]:
#@title 📦 Cell 1: Install Dependencies (Run Once)
#@markdown This installs all required AI libraries. Takes ~2-3 minutes.

print("⏳ Installing dependencies...")

# LESSON LEARNED: Pin versions to avoid compatibility issues
# - gradio>=5.0.0 (NOT 4.x — breaks fastrtc)
# - fastrtc>=0.0.34 (NOT older — missing Stream class)
# - numpy<2.0 (Colab compatibility)

!pip install -q \
    "gradio>=5.0.0" \
    "fastrtc>=0.0.34" \
    "ultralytics>=8.0.0" \
    "tensorflow>=2.12.0" \
    "opencv-python-headless>=4.8.0" \
    "numpy<2.0" \
    "scikit-learn>=1.3.0"

# Verify critical imports
import importlib
checks = {
    "gradio": "gradio",
    "fastrtc (Stream)": "fastrtc",
    "ultralytics (YOLO)": "ultralytics",
    "tensorflow": "tensorflow",
    "cv2": "cv2",
    "sklearn": "sklearn",
}
all_ok = True
for name, mod in checks.items():
    try:
        m = importlib.import_module(mod)
        ver = getattr(m, '__version__', '?')
        print(f"   ✅ {name}: {ver}")
    except ImportError as e:
        print(f"   ❌ {name}: FAILED — {e}")
        all_ok = False

# Critical: verify fastrtc.Stream exists
try:
    from fastrtc import Stream
    print(f"   ✅ fastrtc.Stream: Available")
except ImportError as e:
    print(f"   ❌ fastrtc.Stream: FAILED — {e}")
    all_ok = False

if all_ok:
    print("\n🎉 All dependencies installed successfully!")
    print("👉 Proceed to Cell 2.")
else:
    print("\n⚠️ Some dependencies failed. Try: Runtime → Restart Session, then re-run this cell.")

In [ ]:
#@title 🧠 Cell 2: Setup AI Engine
#@markdown Downloads the YOLO model and loads the trained classifier.

import os
import cv2
import numpy as np
import pickle
import logging
from collections import deque
from pathlib import Path

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger("wasel")

# ─── Clone repo to get the trained model ───
REPO_DIR = "/content/wasel_repo"
if not os.path.exists(REPO_DIR):
    print("⏳ Cloning repository...")
    !git clone -q https://github.com/eltaweelactuary/Wasel_v4_Official_Release.git {REPO_DIR}
    print("   ✅ Repository cloned.")
else:
    print("   ✅ Repository already exists.")

# ─── Load YOLO ───
print("⏳ Loading YOLOv8-Pose model...")
from ultralytics import YOLO
yolo_model = YOLO("yolov8n-pose.pt")  # Auto-downloads if not cached
print("   ✅ YOLOv8n-Pose loaded.")

# ─── Load Trained Classifier ───
classifier = None
label_encoder = None
PKL_PATH = os.path.join(REPO_DIR, "GCP_Source_Code/wasel_v4_data/models/psl_classifier.pkl")

if os.path.exists(PKL_PATH) and os.path.getsize(PKL_PATH) > 100:
    with open(PKL_PATH, 'rb') as f:
        classifier, label_encoder = pickle.load(f)
    print(f"   ✅ PSL Classifier loaded ({os.path.getsize(PKL_PATH)//1024} KB)")
else:
    print(f"   ⚠️ Classifier not found at {PKL_PATH}. Predictions will show 'No model'.")

# ─── Frame Processing Pipeline ───
sequence_buffer = deque(maxlen=30)

def extract_keypoints(frame):
    """Extract YOLO-Pose keypoints from a frame."""
    results = yolo_model(frame, verbose=False)
    if not results or not results[0].keypoints or len(results[0].keypoints.data) == 0:
        return None
    kp = results[0].keypoints.data[0].cpu().numpy()  # (17, 3)
    return kp.flatten()  # (51,)

def predict_sign(sequence):
    """Predict sign from keypoint sequence using sklearn classifier."""
    if classifier is None or len(sequence) < 5:
        return None, 0.0
    features = np.mean(np.array(list(sequence)), axis=0).reshape(1, -1)
    expected = classifier.n_features_in_
    actual = features.shape[1]
    if actual > expected:
        features = features[:, :expected]
    elif actual < expected:
        features = np.pad(features, ((0, 0), (0, expected - actual)))
    probs = classifier.predict_proba(features)[0]
    idx = np.argmax(probs)
    label = label_encoder.inverse_transform([idx])[0]
    return label, float(probs[idx]) * 100

def process_frame(image):
    """Main frame processor for the WebRTC stream."""
    if image is None:
        return None
    try:
        # Run YOLO Pose and draw skeleton
        results = yolo_model(image, verbose=False)
        annotated = results[0].plot()

        # Extract keypoints for prediction
        keypoints = extract_keypoints(image)
        label_text = "Waiting for signs..."

        if keypoints is not None:
            sequence_buffer.append(keypoints)
            if len(sequence_buffer) > 5:
                label, conf = predict_sign(sequence_buffer)
                if label and conf > 45.0:
                    label_text = f"{label} ({conf:.1f}%)"

        # Draw prediction
        cv2.putText(annotated, label_text, (30, 50),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 255, 0), 3)

        # Motion energy detection
        if len(sequence_buffer) >= 2:
            seq = np.array(list(sequence_buffer))
            n_cols = min(seq.shape[1], 126)
            hand_motion = np.diff(seq[:, :n_cols], axis=0)
            energy = np.linalg.norm(hand_motion)
            if energy > 2.0:
                cv2.putText(annotated, "Signing Detected", (30, 90),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)

        return annotated
    except Exception as e:
        logger.error(f"Frame error: {e}")
        return image

print("\n🚀 AI Engine ready!")
print("👉 Proceed to Cell 3 to launch the live demo.")

In [ ]:
#@title 🎥 Cell 3: Launch Live Demo
#@markdown This starts the WebRTC stream and gives you a **public URL**.
#@markdown Share the `*.gradio.live` link with anyone!

from fastrtc import Stream

stream = Stream(
    handler=process_frame,
    modality="video",
    mode="send-receive",
)

print("🎉 Launching Wasel v4 Live Demo...")
print("📋 A public URL will appear below — share it with the client!")
print("⚠️  Keep this cell running. Closing it stops the demo.\n")

# share=True generates a public *.gradio.live URL (valid for 72 hours)
stream.ui.launch(share=True, quiet=False)